# Phase 2 (L-NER) - fine-tuning

Each run loads a Phase 1 checkpoint, adds a BIO token-classification head (softmax, constrained Viterbi or CRF), adapts it (`full`, `partial`, `lora`, `lorma`, `bitfit`, `kd`), keeps the best epoch by strict macro-F1 on held-out documents and predicts the test set with overlapping windows.

**Inputs:** the secret `HF` and one Kaggle dataset per Phase 1 model (weights, config, `bpe_tokenizer.json`), paths in `MODEL_DIRS`. **Output:** `/kaggle/working/phase2_results_bundle.zip` (logs, metrics, submissions, no weights).

`RUNS` lists every experiment of the phase; `RUN_ONLY` selects the runs behind the best submission (`None` = all). Hyperparameters are starting values, not tuned.

In [ ]:
# ============================ CONFIG ============================
MODEL_DIRS = {          # folder (or any file inside it) of each Phase 1 checkpoint; None = auto-discover
    "m2": None,         # Model 2: A-3 decoder, RoPE + GQA          (a3_config.json + a3_model.pt)
    "m3": None,         # Model 3: A-2 GRU encoder-decoder           (gru_config.json + gru_model.pt, or Trainer *.safetensors)
    "m4": "/kaggle/input/datasets/sanskaars22/model-5",   # Model 4: modern decoder (this upload is legal-modern-dec-101m)
    "m5": "/kaggle/input/datasets/sanskaars22/model-4-prefix-lm-labelled-as-m5-in-notebook",   # Model 5: prefix-LM (legal-prefix-lm-101m)
}
DISCOVER_KEYWORDS = {"m2": ["model-2", "model_2", "model2", "dec-rope"], "m3": ["model-3", "model_3", "model3", "gru"],
                     "m4": ["model-4", "model_4", "model4"], "m5": ["model-5", "model_5", "model5", "prefix"]}

COMMON = dict(
    window=1024, stride=512,       # tokens per window / step between windows (overlap = window - stride)
    preamble_tokens=0,             # >0: every window also gets the document's first N tokens in front, as context only
                                   # (no loss, not predicted) - the preamble names the parties, counsel and judges;
                                   # the content part of a window shrinks to window - preamble_tokens
    epochs=8, batch_size=4, grad_accum=2,
    lr_backbone=3e-5,              # pretrained weights (full / partial / kd)
    lr_adapter=3e-4,               # LoRA / LoRMA / BitFit parameters
    lr_head=1e-3,                  # the new classification head
    weight_decay=0.01, warmup_frac=0.1, grad_clip=1.0,
    rank=8, alpha=16, adapter_targets="attn+mlp",   # LoRA / LoRMA
    partial_trainable_frac=0.5,    # partial: train the top half of the layers (+ head)
    head_dropout=0.1,
    class_weighting="none",        # "none" or "sqrt_inv" (inverse-sqrt tag frequency, capped at 10)
    attn="bidir",                  # decoders: "bidir" (no causal mask during fine-tuning) or "causal"
    head="linear",                 # "linear" (per-token softmax) or "crf" (linear-chain CRF with learned tag transitions)
    decode="argmax",               # linear head: "argmax" (per token) or "viterbi" (best BIO-valid tag path); crf always viterbi
    train_on_val=False,            # True: also train on the VAL_DOC_COUNT validation docs and keep the LAST epoch
                                   # (no model selection is possible; set `epochs` yourself; val scores become optimistic)
    kd_temperature=2.0, kd_alpha=0.5,
    student_layers=None,           # keep only the lowest N layers of the checkpoint (any method; None = all layers)
    patience=None,                 # stop after this many epochs without validation improvement (None = off)
    save_weights=True,             # save the trainable parameters of the best epoch
    max_train_windows=None,        # debug: cap windows per epoch
    seed=42,
)

RUNS = [
    dict(name="m4_full_crf",          model="m4", method="full", head="crf"),
    dict(name="m4_kd_student_crf",    model="m4", method="kd", teacher="m4_full_crf", student_layers=4, head="crf"),
    dict(name="m4_lorma_crf_pre",     model="m4", method="lorma", head="crf", preamble_tokens=256),
    dict(name="m5_lorma_crf_pre",     model="m5", method="lorma", head="crf", preamble_tokens=256),
    # final runs: the best setup trained on all 70 documents (no validation -> set epochs yourself) ...
    dict(name="m5_lorma_crf_pre_all", model="m5", method="lorma", head="crf", preamble_tokens=256, train_on_val=True, epochs=...),
    # ... and the same setup with other seeds, for a fixed seed ensemble (with m5_lorma_crf_pre, seed 42)
    dict(name="m5_lorma_crf_pre_s1",  model="m5", method="lorma", head="crf", preamble_tokens=256, seed=1),
    dict(name="m5_lorma_crf_pre_s2",  model="m5", method="lorma", head="crf", preamble_tokens=256, seed=2),
]
RUN_ONLY = ["m5_lorma_crf_pre"]   # runs behind the best submission; None = every run in RUNS
VAL_DOC_COUNT = 10         # whole documents held out from fold1+fold2 for model selection
SMOKE = False              # True: every selected run does 1 epoch on 16 windows (~2-5 min each) to check that
                           # loading, training, decoding and zipping work; outputs go to /kaggle/working/smoke/

In [ ]:
import os, re, json, glob, math, time, random, zipfile, bisect, hashlib, gc, platform
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("p2_work")
if SMOKE: WORK = os.path.join(WORK, "smoke")
os.makedirs(WORK, exist_ok=True)
LOG_PATH = os.path.join(WORK, "phase2_finetune_log.txt")
_log = open(LOG_PATH, "a", encoding="utf-8")

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True); _log.write(s + "\n"); _log.flush()

def section(t):
    log("\n" + "=" * 100); log(t); log("=" * 100)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
section(f"Phase 2 fine-tuning - {time.strftime('%Y-%m-%d %H:%M:%S')}")
log(f"torch {torch.__version__} | python {platform.python_version()} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(0)}, {torch.cuda.device_count()} GPU)" if DEVICE.type == "cuda" else "")
    + f" | amp {AMP_DTYPE}")
log("COMMON =", json.dumps(COMMON))

In [ ]:
# ---- data: Phase_2_LNER from the official dataset ----
def load_phase2_files():
    local = os.environ.get("P2_LOCAL_DATA")                       # local testing only
    if local:
        return {f: os.path.join(local, f) for f in os.listdir(local)}
    from huggingface_hub import HfApi, hf_hub_download
    token, why = os.environ.get("HF_TOKEN"), "HF_TOKEN env var not set"
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF")
        except Exception as e:
            why = f"Kaggle secret 'HF' not readable ({e!r})"
    assert token, (f"no Hugging Face token: {why}. In the Kaggle editor: Add-ons -> Secrets -> add/attach a secret "
                   "named exactly HF (value = your hf_... token) and make sure its checkbox is ON for this notebook.")
    try:
        log(f"HF login ok: {HfApi().whoami(token=token)['name']}")
    except Exception as e:
        raise RuntimeError(f"the HF token was found but is rejected by Hugging Face ({e!r}); "
                           "create a new read token at huggingface.co/settings/tokens") from None
    repo = "Exploration-Lab/CS-781-Capstone"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset", token=token) if f.startswith("Phase_2_LNER/")]
    return {os.path.basename(f): hf_hub_download(repo, f, repo_type="dataset", token=token) for f in files}

P2 = load_phase2_files()
read = lambda n: json.load(open(P2[n], encoding="utf-8"))
LABELS = [l.strip() for l in open(P2["labels.txt"], encoding="utf-8") if l.strip()]
TAGS = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]     # tag 1+2k = B-label_k, 2+2k = I-label_k
N_TAGS = len(TAGS)
IS_I = np.array([t > 0 and (t - 1) % 2 == 1 for t in range(N_TAGS)])
ALLOWED = np.array([[not IS_I[j] or i in (j - 1, j) for j in range(N_TAGS)] for i in range(N_TAGS)])  # [i, j]: j may follow i
START_OK = ~IS_I                                                      # a document / span cannot start with I-

DOCS, GOLD = {}, {}                                                   # train docs (fold1 + fold2)
for fd in ("fold1", "fold2"):
    T, G = read(f"{fd}-text.json"), read(f"{fd}-gold.json")
    for did, text in T.items():
        assert did not in DOCS, f"doc id {did} appears in both folds"
        DOCS[did] = text
        GOLD[did] = sorted((int(s), int(e), l) for s, e, l, *_ in G.get(did, []))
TEST = read("fold3-text.json")
FOLD_OF = {did: fd for fd in ("fold1", "fold2") for did in read(f"{fd}-text.json")}

# validation split: whole documents, both folds represented, and at least 2 documents with witnesses (WIT is rare)
_rng = random.Random(COMMON["seed"])
_ids = sorted(DOCS)
_wit = [d for d in _ids if any(l == "WIT" for _, _, l in GOLD[d])]
VAL_IDS = set()
for fd in ("fold1", "fold2"):
    VAL_IDS.add(_rng.choice([d for d in _wit if FOLD_OF[d] == fd] or _wit))
for fd in ("fold1", "fold2") * VAL_DOC_COUNT:
    if len(VAL_IDS) >= VAL_DOC_COUNT: break
    VAL_IDS.add(_rng.choice([d for d in _ids if FOLD_OF[d] == fd and d not in VAL_IDS]))
TRAIN_IDS = [d for d in _ids if d not in VAL_IDS]
VAL_IDS = sorted(VAL_IDS)

section("DATA")
log(f"labels: {LABELS}")
log(f"train docs {len(TRAIN_IDS)}, val docs {len(VAL_IDS)}, test docs {len(TEST)} "
    f"(empty test docs: {sum(1 for t in TEST.values() if not t.strip())})")
log(f"val ids: {VAL_IDS}")
for name, ids in (("train", TRAIN_IDS), ("val", VAL_IDS)):
    c = Counter(l for d in ids for _, _, l in GOLD[d])
    log(f"{name:5s} entities: " + ", ".join(f"{l}={c[l]}" for l in LABELS))

In [ ]:
# ---- Phase 1 byte-level BPE (same algorithm and ids as LegalBPETokenizer), with character offsets ----
WORD_RE = re.compile(r"\S+")
_SHIFT = 20
_EOW = (1 << _SHIFT) - 1

class OffsetBPE:
    def __init__(self, path):
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        self.itos = d["itos"]
        sym2id = {tuple(s): i for i, s in enumerate(self.itos) if isinstance(s, list)}
        to_id = lambda t: _EOW if tuple(t) == (256,) else sym2id[tuple(t)]
        self.ranks = {(to_id(l) << _SHIFT) | to_id(r): sym2id[tuple(l) + tuple(r)] for l, r in d["merges"]}
        self.nbytes = [sum(b != 256 for b in s) if isinstance(s, list) else 0 for s in self.itos]
        self.eow_id = len(self.itos)               # LegalBPETokenizer: a bare end-of-word symbol gets id len(itos)
        self.vocab_size = len(self.itos) + 1
        self.cache = {}

    def tokenize_word(self, word):
        out = self.cache.get(word)
        if out is not None: return out
        w = [b + 4 for b in word.encode("utf-8")] + [_EOW]
        ranks = self.ranks
        while len(w) > 1:
            best = None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best is None or n < best[0]): best = (n, x, y)
            if best is None: break
            n, a, b = best; i, res = 0, []
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: res.append(n); i += 2
                else: res.append(w[i]); i += 1
            w = res
        self.cache[word] = w
        return w

    def encode(self, text):
        """-> ids, char_start, char_end (np arrays). A bare end-of-word token has an empty span at the word end."""
        ids, cs, ce = [], [], []
        for m in WORD_RE.finditer(text):
            word, base = m.group(), m.start()
            b2c = []
            for ci, ch in enumerate(word): b2c.extend([ci] * len(ch.encode("utf-8")))
            p = 0
            for t in self.tokenize_word(word):
                nb = 0 if t == _EOW else self.nbytes[t]
                ids.append(self.eow_id if t == _EOW else t)
                if nb == 0:
                    cs.append(base + len(word)); ce.append(base + len(word))
                else:
                    cs.append(base + b2c[p]); ce.append(base + b2c[p + nb - 1] + 1); p += nb
        return np.array(ids, np.int64), np.array(cs, np.int64), np.array(ce, np.int64)

def token_tags(cs, ce, ents):
    """BIO tag per token: a token overlapping an entity is B (first) / I (rest); empty tokens -> -100 (no loss)."""
    tags = np.zeros(len(cs), np.int64)
    tags[ce <= cs] = -100
    for s, e, l in ents:
        k = LABELS.index(l)
        i = bisect.bisect_right(ce, s); first = True
        while i < len(cs) and cs[i] < e:
            if ce[i] > cs[i]:
                tags[i] = 1 + 2 * k if first else 2 + 2 * k; first = False
            i += 1
    return tags

_TOKENIZERS, _ENCODINGS = {}, {}
def get_encodings(tok_path):
    """Tokenizer + encoded train/test docs, cached per tokenizer file content."""
    h = hashlib.md5(open(tok_path, "rb").read()).hexdigest()
    if h not in _ENCODINGS:
        t0 = time.time(); bpe = OffsetBPE(tok_path)
        enc = {}
        for did, text in list(DOCS.items()) + list(TEST.items()):
            ids, cs, ce = bpe.encode(text)
            enc[did] = dict(ids=ids, cs=cs, ce=ce, tags=token_tags(cs, ce, GOLD[did]) if did in GOLD else None)
        _TOKENIZERS[h], _ENCODINGS[h] = bpe, enc
        n = [len(enc[d]["ids"]) for d in DOCS]
        log(f"tokenizer {tok_path} (md5 {h[:8]}, vocab {bpe.vocab_size}): encoded {len(enc)} docs in {time.time() - t0:.0f}s, "
            f"train tokens/doc median {int(np.median(n))}, max {max(n)}")
    return _TOKENIZERS[h], _ENCODINGS[h]

In [ ]:
# ---- spans: decoding, boundary trimming, scoring (= evaluate_phase2.py) ----
EDGE = set(" .,;:()[]{}\"'`‘’“”")
ABBR_RE = re.compile(r"(?:[A-Z]\.)+")              # initials: J.  U.P.  S.K.  (other abbreviations are learned)

def tags_to_spans(tags, cs, ce):
    """Token tags -> (start, end, label_index). I- after O or after another label starts a new span."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        if ce[i] <= cs[i]: continue                      # empty end-of-word token: neither breaks nor extends
        if t <= 0:
            if cur: spans.append(tuple(cur)); cur = None
            continue
        k, is_b = (t - 1) // 2, (t - 1) % 2 == 0
        if cur is not None and not is_b and cur[2] == k:
            cur[1] = int(ce[i])
        else:
            if cur: spans.append(tuple(cur))
            cur = [int(cs[i]), int(ce[i]), k]
    if cur: spans.append(tuple(cur))
    return spans

def trim(text, s, e, policy, keep_dot=frozenset()):
    """Token spans include punctuation glued to the word ("Ghosh," "IPC.)"); cut it off the edges.
    basic: strip every edge punctuation char. smart: keep a '.' that ends an abbreviation, keep a ')' / ']' that
    closes a bracket opened inside the span, and drop a trailing possessive 's."""
    if policy == "none": return s, e
    while s < e and text[s] in EDGE: s += 1
    while e > s:
        ch = text[e - 1]
        if policy == "smart":
            if text[s:e].endswith(("'s", "’s")) and e - s > 2: e -= 2; continue
            if ch == ".":
                last = text[s:e].split()[-1]
                if last in keep_dot or ABBR_RE.fullmatch(last): break
            if ch == ")" and text[s:e].count("(") >= text[s:e].count(")"): break
            if ch == "]" and text[s:e].count("[") >= text[s:e].count("]"): break
        if ch in EDGE: e -= 1
        else: break
    return s, e

def decode_doc(tags, enc, text, policy, keep_dot):
    out = set()
    for s, e, k in tags_to_spans(tags, enc["cs"], enc["ce"]):
        s, e = trim(text, s, e, policy, keep_dot)
        if e > s: out.add((s, e, LABELS[k]))
    return sorted(out)

def score(gold, pred, ids):
    st = {l: [0, 0, 0] for l in LABELS}
    for d in ids:
        g, p = set(gold[d]), set(pred.get(d, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per = {}
    for l, (tp, fp, fn) in st.items():
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = dict(p=pr, r=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, tp=tp, fp=fp, fn=fn)
    m = lambda k: sum(v[k] for v in per.values()) / len(per)
    return dict(macro_f1=m("f1"), macro_p=m("p"), macro_r=m("r"), per_label=per)

def error_breakdown(gold, pred, ids):
    """Where the strict errors come from (per gold label): wrong boundary, wrong label, both, or missed; and spurious predictions."""
    c = defaultdict(Counter)
    for d in ids:
        g, p = set(gold[d]), set(pred.get(d, []))
        for (s, e, l) in g - p:
            ov = [(ps, pe, pl) for (ps, pe, pl) in p if pe > s and ps < e]
            if not ov: c[l]["missed"] += 1
            elif any((ps, pe) == (s, e) for ps, pe, _ in ov): c[l]["right_span_wrong_label"] += 1
            elif any(pl == l for _, _, pl in ov): c[l]["right_label_wrong_boundary"] += 1
            else: c[l]["wrong_label_and_boundary"] += 1
        for (s, e, l) in p - g:
            if not any(ge > s and gs < e for gs, ge, _ in g): c[l]["spurious_pred"] += 1
    return {l: dict(c[l]) for l in LABELS}

def learn_keep_dot(ids):
    """Words ending in '.' that the annotators keep at the end of an entity more often than not (from training gold)."""
    keep, total = Counter(), Counter()
    for d in ids:
        text = DOCS[d]
        for s, e, l in GOLD[d]:
            if not text[s:e].split(): continue
            ent_last = text[s:e].split()[-1]
            if ent_last.endswith("."): keep[ent_last] += 1; total[ent_last] += 1
            elif text[e:e + 1] == ".": total[ent_last + "."] += 1
    return frozenset(w for w in total if keep[w] * 2 > total[w])

In [ ]:
# ---- From-scratch attention (no nn.MultiheadAttention / F.scaled_dot_product_attention), as in Phase 1 ----
# out = softmax(q k^T * scale, masked) v, computed in query blocks with the softmax statistics kept per row
# (log-sum-exp), so the full (T, S) score matrix is never stored. The backward pass is written by hand and
# recomputes each block's probabilities from the saved log-sum-exp:
#   P = exp(S - lse),  dV = P^T dO,  dP = dO V^T,  dS = P * (dP - rowsum(dO * O)),  dQ = dS K * scale,  dK = dS^T Q * scale
def _block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset):
    s = torch.matmul(q_blk, k[:, :, :kend].transpose(-2, -1)).float() * scale
    if attn_mask is not None:
        s = s.masked_fill(~attn_mask[..., i:j, :kend], float("-inf"))
    if is_causal:
        rows = torch.arange(i, j, device=s.device)[:, None] + offset
        s = s.masked_fill(torch.arange(kend, device=s.device)[None, :] > rows, float("-inf"))
    return s


class BlockwiseAttention(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k, v, attn_mask, is_causal, scale, block):
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        out = torch.empty_like(q)
        lse = torch.empty(q.shape[:-1], dtype=torch.float32, device=q.device)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            s = _block_scores(q[:, :, i:j], k, scale, attn_mask, is_causal, i, j, kend, offset)
            m = s.amax(-1, keepdim=True)
            p = torch.exp(s - m)
            l = p.sum(-1, keepdim=True)
            out[:, :, i:j] = torch.matmul((p / l).to(v.dtype), v[:, :, :kend])
            lse[:, :, i:j] = (m + l.log()).squeeze(-1)
        ctx.save_for_backward(q, k, v, out, lse, attn_mask)
        ctx.is_causal, ctx.scale, ctx.block = is_causal, scale, block
        return out

    @staticmethod
    def backward(ctx, d_out):
        q, k, v, out, lse, attn_mask = ctx.saved_tensors
        is_causal, scale, block = ctx.is_causal, ctx.scale, ctx.block
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        d_out = d_out.contiguous()
        dq = torch.empty_like(q)
        dk = torch.zeros(k.shape, dtype=torch.float32, device=k.device)
        dv = torch.zeros(v.shape, dtype=torch.float32, device=v.device)
        delta = (d_out.float() * out.float()).sum(-1, keepdim=True)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            q_blk, do_blk = q[:, :, i:j], d_out[:, :, i:j]
            p = torch.exp(_block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset) - lse[:, :, i:j, None])
            dv[:, :, :kend] += torch.matmul(p.to(v.dtype).transpose(-2, -1), do_blk).float()
            dp = torch.matmul(do_blk, v[:, :, :kend].transpose(-2, -1)).float()
            ds = (p * (dp - delta[:, :, i:j])).to(q.dtype)
            dq[:, :, i:j] = torch.matmul(ds, k[:, :, :kend]) * scale
            dk[:, :, :kend] += torch.matmul(ds.transpose(-2, -1), q_blk).float() * scale
        return dq, dk.to(k.dtype), dv.to(v.dtype), None, None, None, None


def scratch_attention(q, k, v, attn_mask=None, is_causal=False, scale=None, block=128):
    """q (B, H, T, d), k/v (B, H_kv, S, d) with H_kv dividing H (GQA). attn_mask: bool (True = may attend)."""
    scale = q.size(-1) ** -0.5 if scale is None else scale
    B, H, T, d = q.shape
    n_rep = H // k.size(1)
    if n_rep > 1 and not is_causal:
        q = q.reshape(B, k.size(1), n_rep * T, d)
        if attn_mask is not None:
            attn_mask = attn_mask.repeat(1, 1, n_rep, 1)
        out = BlockwiseAttention.apply(q, k, v, attn_mask, False, scale, block)
        return out.reshape(B, H, T, d)
    if n_rep > 1:
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
    return BlockwiseAttention.apply(q, k, v, attn_mask, is_causal, scale, block)


def attend(q, k, v, key_keep, causal, scale=None):
    """causal or bidirectional attention; key_keep (B, T) bool marks real tokens (None = no padding)."""
    if key_keep is None:
        return scratch_attention(q, k, v, is_causal=causal, scale=scale)
    T = q.size(-2)
    allowed = key_keep[:, None, None, :].expand(-1, 1, T, -1)
    if causal:
        allowed = allowed & torch.ones(T, T, dtype=torch.bool, device=q.device).tril()
    allowed = allowed | torch.eye(T, dtype=torch.bool, device=q.device)   # padding rows see themselves (no NaN)
    return scratch_attention(q, k, v, attn_mask=allowed, scale=scale)


def rope_cache(T, d_head, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))
    ang = torch.outer(torch.arange(T, dtype=torch.float32), inv_freq)
    ang = torch.cat([ang, ang], dim=-1)
    return ang.cos(), ang.sin()

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

In [ ]:
# ---- Phase 1 backbones as encoders. Parameter names match the Phase 1 checkpoints exactly. ----
class A3Attention(nn.Module):              # Model 2 (A-3 CausalSelfAttention: MHA/GQA, RoPE or sinusoidal)
    def __init__(self, c):
        super().__init__()
        self.n_heads, self.n_kv, self.d_head = c["n_heads"], c["n_kv_heads"], c["d_model"] // c["n_heads"]
        self.scale = c.get("attention_scale") or self.d_head ** -0.5
        self.q_proj = nn.Linear(c["d_model"], self.n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.v_proj = nn.Linear(c["d_model"], self.n_kv * self.d_head, bias=False)
        self.out_proj = nn.Linear(self.n_heads * self.d_head, c["d_model"], bias=False)
        self.use_rope = c.get("pos_emb_type") == "rope"
        if self.use_rope:
            cos, sin = rope_cache(c["max_seq_len"], self.d_head, c.get("rope_base", 10000.0))
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, key_keep, causal):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.d_head).transpose(1, 2)
        if self.use_rope:
            cos, sin = self.rope_cos[:T][None, None].to(q.dtype), self.rope_sin[:T][None, None].to(q.dtype)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        out = attend(q, k, v, key_keep, causal, self.scale)
        return self.out_proj(out.transpose(1, 2).reshape(B, T, -1))


class A3Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.norm_first = c.get("norm_first", False)
        self.ln1 = nn.LayerNorm(c["d_model"])
        self.attn = A3Attention(c)
        self.ln2 = nn.LayerNorm(c["d_model"])
        self.ffn = nn.Module()
        self.ffn.net = nn.Sequential(nn.Linear(c["d_model"], c["d_ff"]), nn.GELU(), nn.Linear(c["d_ff"], c["d_model"]), nn.Dropout(0.0))

    def forward(self, x, key_keep, causal):
        if self.norm_first:
            x = x + self.attn(self.ln1(x), key_keep, causal)
            return x + self.ffn.net(self.ln2(x))
        x = self.ln1(x + self.attn(x, key_keep, causal))
        return self.ln2(x + self.ffn.net(x))


class A3Backbone(nn.Module):
    kind = "a3"
    def __init__(self, c):
        super().__init__()
        assert c.get("attention_type", "standard") == "standard", "only the standard (MHA/GQA) A-3 attention is supported"
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        if c.get("pos_emb_type") == "sinusoidal":
            pos = torch.arange(c["max_seq_len"], dtype=torch.float32)[:, None]
            div = torch.exp(torch.arange(0, c["d_model"], 2, dtype=torch.float32) * (-math.log(10000.0) / c["d_model"]))
            pe = torch.zeros(c["max_seq_len"], c["d_model"]); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
            self.register_buffer("pos_emb", pe, persistent=False)
        self.blocks = nn.ModuleList([A3Block(c) for _ in range(c["n_layers"])])
        self.ln_f = nn.LayerNorm(c["d_model"])

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        if hasattr(self, "pos_emb"): x = x + self.pos_emb[:ids.size(1)][None]
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.ln_f(x)


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).to(x.dtype) * self.weight


class ModernAttention(A3Attention):         # Models 4/5 GQAttention (always RoPE), same projections
    def __init__(self, c):
        super().__init__({**c, "pos_emb_type": "rope", "attention_scale": None})


class ModernBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.attn = ModernAttention(c)
        self.mlp_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))
        self.mlp = nn.Module()
        self.mlp.gate = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.up = nn.Linear(c["d_model"], c["d_ff"], bias=False)
        self.mlp.down = nn.Linear(c["d_ff"], c["d_model"], bias=False)

    def forward(self, x, key_keep, causal):
        x = x + self.attn(self.attn_norm(x), key_keep, causal)
        h = self.mlp_norm(x)
        return x + self.mlp.down(F.silu(self.mlp.gate(h)) * self.mlp.up(h))


class ModernBackbone(nn.Module):
    kind = "modern"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, c["d_model"]
        self.tok_emb = nn.Embedding(c["vocab_size"], c["d_model"])
        self.blocks = nn.ModuleList([ModernBlock(c) for _ in range(c["n_layers"])])
        self.final_norm = RMSNorm(c["d_model"], c.get("norm_eps", 1e-5))

    def forward(self, ids, key_keep=None, causal=False):
        x = self.tok_emb(ids)
        for b in self.blocks: x = b(x, key_keep, causal)
        return self.final_norm(x)


class GRUBackbone(nn.Module):               # Model 3: the bidirectional GRU encoder of the A-2 seq2seq model
    kind = "gru"
    def __init__(self, c):
        super().__init__()
        self.c, self.hidden_size = c, 2 * c["hidden"]
        self.embedding = nn.Embedding(c["vocab_size"], c["emb_dim"])
        self.enc_rnn = nn.GRU(c["emb_dim"], c["hidden"], num_layers=c["enc_layers"], batch_first=True, bidirectional=True)

    def forward(self, ids, key_keep=None, causal=False):
        self.enc_rnn.flatten_parameters()
        x = self.embedding(ids)
        if key_keep is None:
            return self.enc_rnn(x)[0]
        lengths = key_keep.sum(1).clamp(min=1).cpu()
        out, _ = self.enc_rnn(pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False))
        return pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))[0]

In [ ]:
# ---- finding and loading the checkpoints ----
def _ckpt_kind(folder):
    names = os.listdir(folder)
    if "a3_config.json" in names: return "a3"
    if "modern_config.json" in names: return "modern"
    if "gru_config.json" in names: return "gru"
    st = [n for n in names if n.endswith(".safetensors")]
    if st:
        from safetensors import safe_open
        with safe_open(os.path.join(folder, st[0]), "pt") as f:
            keys = list(f.keys())
        if any("enc_rnn" in k for k in keys): return "gru"
        if any("attn_norm" in k for k in keys): return "modern"
        if any(".ln1." in k for k in keys): return "a3"
    return None

WANT_KIND = {"m2": "a3", "m3": "gru", "m4": "modern", "m5": "modern"}

def find_checkpoint(key):
    p = MODEL_DIRS.get(key)
    if p:
        folder = p if os.path.isdir(p) else os.path.dirname(p)
        if _ckpt_kind(folder) is None:          # a dataset root: look one level deeper
            subs = [os.path.dirname(f) for f in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                    if f.endswith(("_config.json", ".safetensors"))]
            subs = [s for s in dict.fromkeys(subs) if "checkpoint-" not in s and _ckpt_kind(s)]
            assert subs, f"no checkpoint found under {p}"
            folder = subs[0]
        return folder
    cands = []
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if f.endswith(("_config.json", ".safetensors")):
            folder = os.path.dirname(f)
            if "checkpoint-" in folder: continue
            if _ckpt_kind(folder) == WANT_KIND[key]: cands.append(folder)
    cands = list(dict.fromkeys(cands))
    hit = [c for c in cands if any(w in c.lower() for w in DISCOVER_KEYWORDS[key])]
    shared = sum(k == WANT_KIND[key] for k in WANT_KIND.values()) > 1   # m4/m5: same format, never guess
    assert hit or (len(cands) == 1 and not shared), \
        f"{key}: cannot pick a checkpoint automatically from {cands}; set MODEL_DIRS['{key}']"
    return (hit or cands)[0]

def _strip(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("module.", "lm.", "model."):
            if k.startswith(pre): k = k[len(pre):]
        out[k] = v
    return out

def load_backbone(key, n_layers=None):
    """Phase 1 checkpoint -> (backbone, info). n_layers < full depth keeps only the lowest layers (KD student)."""
    folder = find_checkpoint(key)
    kind = _ckpt_kind(folder)
    names = os.listdir(folder)
    pt = [n for n in names if n.endswith("_model.pt")]
    if pt:
        sd = torch.load(os.path.join(folder, pt[0]), map_location="cpu", weights_only=True)
    else:
        from safetensors.torch import load_file
        sd = load_file(os.path.join(folder, [n for n in names if n.endswith(".safetensors")][0]))
    sd = _strip(sd)
    cfg_name = {"a3": "a3_config.json", "modern": "modern_config.json", "gru": "gru_config.json"}[kind]
    if cfg_name in names:
        c = json.load(open(os.path.join(folder, cfg_name)))
    elif kind == "gru":                          # Trainer checkpoint without config: read the sizes off the weights
        c = dict(vocab_size=sd["embedding.weight"].shape[0], emb_dim=sd["embedding.weight"].shape[1],
                 hidden=sd["enc_rnn.weight_hh_l0"].shape[1],
                 enc_layers=len([k for k in sd if re.fullmatch(r"enc_rnn\.weight_ih_l\d+", k)]))
    else:
        raise FileNotFoundError(f"{folder}: {cfg_name} is needed to know the number of heads")
    full_layers = c["enc_layers"] if kind == "gru" else c["n_layers"]
    if n_layers:
        assert n_layers <= full_layers
        if kind == "gru":
            c = {**c, "enc_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not re.search(r"_l(\d+)", k) or int(re.search(r"_l(\d+)", k).group(1)) < n_layers}
        else:
            c = {**c, "n_layers": n_layers}
            sd = {k: v for k, v in sd.items() if not k.startswith("blocks.") or int(k.split(".")[1]) < n_layers}
    model = {"a3": A3Backbone, "modern": ModernBackbone, "gru": GRUBackbone}[kind](c)
    own = model.state_dict()
    used = {k: v for k, v in sd.items() if k in own}
    missing = [k for k in own if k not in used]
    assert not missing, f"weights missing from the checkpoint: {missing[:5]}"
    for k, v in used.items():
        assert v.shape == own[k].shape, f"{k}: checkpoint {tuple(v.shape)} vs model {tuple(own[k].shape)}"
    model.load_state_dict(used)
    tok = [os.path.join(folder, n) for n in names if n.startswith("bpe_tokenizer") and n.endswith(".json")]
    for where in (os.path.join(os.path.dirname(folder), "**"), "/kaggle/input/**"):     # next to the weights, then anywhere
        tok = tok or sorted(glob.glob(os.path.join(where, "bpe_tokenizer*.json"), recursive=True))
    assert tok, f"no bpe_tokenizer*.json found for {key}: add the Phase 1 tokenizer as an input"
    info = dict(folder=folder, kind=kind, config=c, full_layers=full_layers, n_layers=n_layers or full_layers,
                unused_keys=len(sd) - len(used), tokenizer=tok[0], prefix_lm=c.get("prefix_lm"))
    return model, info

In [ ]:
# ---- adaptation methods ----
class LoRALinear(nn.Module):
    """LoRA (Hu et al. 2021): h = W0 x + (alpha / r) B A x, A (r x in) random, B (out x r) = 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel & Modi 2025): multiplicative update h = (I + (alpha / r) B A) W0 x, with A (r x out) random,
    B (out x r) = 0 so training starts at W0. Computed as z = W0 x, then z + (alpha / r) B (A z): never forms the d x d matrix."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + (z @ self.A.t().to(z.dtype)) @ self.B.t().to(z.dtype) * self.scale
        return z if self.base.bias is None else z + self.base.bias


def adapter_target_names(backbone, targets):
    names = []
    for n, m in backbone.named_modules():
        if not isinstance(m, nn.Linear): continue
        is_attn = ".attn." in n
        is_mlp = ".ffn." in n or ".mlp." in n
        if (is_attn and "attn" in targets) or (is_mlp and "mlp" in targets): names.append(n)
    return names

def inject(backbone, cls, targets, r, alpha):
    names = adapter_target_names(backbone, targets)
    for n in names:
        parent, child = backbone.get_submodule(n.rsplit(".", 1)[0]), n.rsplit(".", 1)[1]
        setattr(parent, child, cls(getattr(parent, child), r, alpha))
    return names


class CRF(nn.Module):
    """Linear-chain CRF over the BIO tags. Transitions that break BIO (I-x not after B-x / I-x) are fixed at -1e4.
    Positions tagged -100 (padding, empty end-of-word tokens) are skipped: the chain links the remaining tokens."""
    def __init__(self, n):
        super().__init__()
        self.trans = nn.Parameter(torch.zeros(n, n))           # trans[i, j]: score of tag j right after tag i
        self.start = nn.Parameter(torch.zeros(n)); self.end = nn.Parameter(torch.zeros(n))
        self.register_buffer("pen_t", torch.from_numpy(np.where(ALLOWED, 0.0, -1e4)).float())
        self.register_buffer("pen_s", torch.from_numpy(np.where(START_OK, 0.0, -1e4)).float())

    def scores(self):
        return self.trans + self.pen_t, self.start + self.pen_s, self.end

    def nll(self, em, tags):
        """em (B, T, n) float32, tags (B, T). Negative log-likelihood summed over windows / number of tagged tokens
        (so its scale matches the per-token cross-entropy)."""
        trans, start, end = self.scores()
        B, T, n = em.shape
        mask = tags != -100
        y = tags.clamp(min=0)
        pos = torch.arange(T, device=em.device).expand(B, T)
        seen = torch.where(mask, pos, torch.full_like(pos, -1)).cummax(1).values      # last tagged position <= t
        prev = torch.cat([torch.full_like(seen[:, :1], -1), seen[:, :-1]], 1)          # last tagged position < t
        y_prev = y.gather(1, prev.clamp(min=0))
        gold = ((em.gather(2, y[..., None]).squeeze(-1) + torch.where(prev >= 0, trans[y_prev, y], start[y])) * mask).sum(1)
        gold = gold + end[y.gather(1, seen[:, -1:].clamp(min=0)).squeeze(1)]
        # partition function as a product of per-token (n x n) score matrices in the log semiring, reduced pairwise
        # (log2 T parallel steps instead of T sequential ones). Untagged positions are the identity; the first tagged
        # token uses `start` (every row the same) and the chain starts from state 0.
        first = mask & (prev < 0)
        M = torch.where(first[..., None, None], start + em[:, :, None, :], trans + em[:, :, None, :])
        eye = torch.full((n, n), -1e4, device=em.device).fill_diagonal_(0.0)
        M = torch.where(mask[..., None, None], M, eye)
        while M.shape[1] > 1:
            if M.shape[1] % 2: M = torch.cat([M, eye.expand(B, 1, n, n)], 1)
            M = (M[:, 0::2, :, :, None] + M[:, 1::2, None, :, :]).logsumexp(-2)
        log_z = torch.logsumexp(M[:, 0, 0] + end, -1)
        has = mask.any(1)
        return (log_z - gold)[has].sum() / mask.sum().clamp(min=1)


class Tagger(nn.Module):
    def __init__(self, backbone, n_tags, dropout, causal, head="linear"):
        super().__init__()
        self.backbone, self.causal = backbone, causal
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.hidden_size, n_tags)
        self.crf = CRF(n_tags) if head == "crf" else None

    def forward(self, ids, key_keep=None):
        h = self.backbone(ids, key_keep, self.causal)
        return self.head(self.drop(h)).float()

    def decoder(self, decode):
        """None = per-token argmax; else (trans, start, end, normalize) numpy arrays for viterbi()."""
        if self.crf is not None:
            return tuple(x.detach().float().cpu().numpy() for x in self.crf.scores()) + (False,)
        if decode == "viterbi":
            return np.where(ALLOWED, 0.0, -1e4), np.where(START_OK, 0.0, -1e4), np.zeros(N_TAGS), True
        return None


def setup_method(model, cfg):
    """Freezes / wraps parameters for the run's method. Returns a list of (param group name, params)."""
    bb, method = model.backbone, cfg["method"]
    for p in bb.parameters(): p.requires_grad = method in ("full", "kd", "partial")
    adapters = []
    if method == "partial":                          # train the top layers only
        (bb.embedding if bb.kind == "gru" else bb.tok_emb).requires_grad_(False)
        if bb.kind == "gru":
            L = bb.c["enc_layers"]; keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for n, p in bb.enc_rnn.named_parameters():
                if int(re.search(r"_l(\d+)", n).group(1)) < keep_from: p.requires_grad = False
        else:
            L = len(bb.blocks); keep_from = L - max(1, round(L * cfg["partial_trainable_frac"]))
            for b in bb.blocks[:keep_from]: b.requires_grad_(False)
        log(f"partial: layers {keep_from}..{L - 1} trainable")
    elif method in ("lora", "lorma"):
        assert bb.kind != "gru", "LoRA/LoRMA are implemented for Linear layers; the GRU encoder has none"
        names = inject(bb, LoRALinear if method == "lora" else LoRMALinear, cfg["adapter_targets"], cfg["rank"], cfg["alpha"])
        adapters = [p for n, p in bb.named_parameters() if n.endswith((".A", ".B"))]
        log(f"{method}: rank {cfg['rank']}, alpha {cfg['alpha']}, {len(names)} adapted Linear layers, e.g. {names[:4]}")
    elif method == "bitfit":
        adapters = [p for n, p in bb.named_parameters() if n.endswith("bias") or re.search(r"bias_(ih|hh)", n)]
        assert adapters, "this backbone has no bias terms: BitFit is not applicable (Models 4/5 use bias-free layers)"
        for p in adapters: p.requires_grad = True
        log(f"bitfit: {len(adapters)} bias tensors trainable")
    for p in adapters: p.requires_grad = True
    ad_ids = {id(p) for p in adapters}
    backbone_params = [p for p in bb.parameters() if p.requires_grad and id(p) not in ad_ids]
    head = list(model.head.parameters()) + (list(model.crf.parameters()) if model.crf is not None else [])
    return [("head", head, cfg["lr_head"]),
            ("adapter", adapters, cfg["lr_adapter"]),
            ("backbone", backbone_params, cfg["lr_backbone"])]

In [ ]:
# ---- windows, prediction ----
def window_starts(n, W, S):
    if n <= W: return [0]
    st = list(range(0, n - W + 1, S))
    if st[-1] + W < n: st.append(n - W)
    return st

def layout(s, n, W, P):
    """Window whose content starts at token s of an n-token document: the first p = min(P, s) document tokens go in front
    as context (preamble: parties, counsel, judges), then content tokens s .. s + c. Returns (p, c)."""
    p = min(P, s)
    return p, min(W - P, n - s)

def make_batch(items, W, enc, teacher=None, P=0):
    """items: list of (doc_id, content start). Returns ids, key_keep (None if no padding), tags, teacher logits.
    Context (preamble) positions get tag -100: no loss, no KD, never predicted."""
    B = len(items)
    ids = torch.zeros(B, W, dtype=torch.long); keep = torch.zeros(B, W, dtype=torch.bool)
    tags = torch.full((B, W), -100, dtype=torch.long)
    tl = torch.zeros(B, W, N_TAGS) if teacher is not None else None
    for i, (d, s) in enumerate(items):
        e = enc[d]; p, c = layout(s, len(e["ids"]), W, P)
        ids[i, :p] = torch.from_numpy(e["ids"][:p])
        ids[i, p:p + c] = torch.from_numpy(e["ids"][s:s + c]); keep[i, :p + c] = True
        if e["tags"] is not None: tags[i, p:p + c] = torch.from_numpy(e["tags"][s:s + c])
        if teacher is not None: tl[i, p:p + c] = teacher[d][s:s + c].float()
    return ids, (None if bool(keep.all()) else keep), tags, tl

@torch.no_grad()
def doc_logits(model, enc, doc_ids, W, S, batch_size=8, P=0):
    """Per-token logits for whole documents: windows overlap, overlapping predictions are averaged."""
    model.eval()
    out = {}
    items = [(d, s) for d in doc_ids if len(enc[d]["ids"]) for s in window_starts(len(enc[d]["ids"]), W - P, S)]
    acc = {d: torch.zeros(len(enc[d]["ids"]), N_TAGS) for d in doc_ids}
    cnt = {d: torch.zeros(len(enc[d]["ids"]), 1) for d in doc_ids}
    for i in range(0, len(items), batch_size):
        chunk = items[i:i + batch_size]
        ids, keep, _, _ = make_batch(chunk, W, enc, P=P)
        with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
            lg = model(ids.to(DEVICE), None if keep is None else keep.to(DEVICE)).float().cpu()
        for j, (d, s) in enumerate(chunk):
            p, c = layout(s, len(enc[d]["ids"]), W, P)
            acc[d][s:s + c] += lg[j, p:p + c]; cnt[d][s:s + c] += 1
    for d in doc_ids:
        out[d] = acc[d] / cnt[d].clamp(min=1)
    return out

def viterbi(em, trans, start, end):
    """Highest-scoring tag path. em (n, N_TAGS) numpy; trans[i, j] = score of j after i."""
    n = len(em)
    bp = np.zeros((n, N_TAGS), np.int64)
    sc = start + em[0]
    for t in range(1, n):
        m = sc[:, None] + trans
        bp[t] = m.argmax(0); sc = m.max(0) + em[t]
    path = np.zeros(n, np.int64); path[-1] = (sc + end).argmax()
    for t in range(n - 1, 0, -1): path[t - 1] = bp[t, path[t]]
    return path

def doc_tags(lg, e, dec):
    if dec is None: return lg.argmax(-1).numpy()
    trans, start, end, normalize = dec
    ne = e["ce"] > e["cs"]                           # empty end-of-word tokens are left out of the chain
    tags = np.zeros(len(lg), np.int64)
    if ne.any():
        em = (F.log_softmax(lg, -1) if normalize else lg)[torch.from_numpy(ne)].numpy()
        tags[ne] = viterbi(em, trans, start, end)
    return tags

def predict_spans(logits, enc, texts, policy, keep_dot, dec=None):
    """dec: Tagger.decoder(...) (None = per-token argmax)."""
    return {d: (decode_doc(doc_tags(logits[d], enc[d], dec), enc[d], texts[d], policy, keep_dot) if len(logits[d]) else [])
            for d in logits}

In [ ]:
# ---- boundary-trim policy: chosen on the TRAIN documents by encoding gold -> tags -> decoding -> scoring ----
section("BOUNDARY TRIM POLICY (gold round trip on the training documents)")
_tok0 = sorted(glob.glob("/kaggle/input/**/bpe_tokenizer*.json", recursive=True))
_bpe0, _enc0 = get_encodings(_tok0[0] if _tok0 else load_backbone("m4")[1]["tokenizer"])
KEEP_DOT = learn_keep_dot(TRAIN_IDS)
log(f"learned abbreviations kept with their final '.': {len(KEEP_DOT)}, e.g. {sorted(KEEP_DOT)[:15]}")
_ceil = {}
for pol in ("none", "basic", "smart"):
    for name, ids in (("train", TRAIN_IDS), ("val", VAL_IDS)):
        pred = {d: decode_doc(_enc0[d]["tags"], _enc0[d], DOCS[d], pol, KEEP_DOT) for d in ids}
        _ceil[(pol, name)] = score(GOLD, pred, ids)["macro_f1"]
    log(f"  policy {pol:6s}: macro-F1 ceiling train {_ceil[(pol, 'train')]:.4f} | val {_ceil[(pol, 'val')]:.4f}")
TRIM_POLICY = max(("none", "basic", "smart"), key=lambda p: _ceil[(p, "train")])
_r = score(GOLD, {d: decode_doc(_enc0[d]["tags"], _enc0[d], DOCS[d], TRIM_POLICY, KEEP_DOT) for d in TRAIN_IDS}, TRAIN_IDS)
log(f"chosen policy: {TRIM_POLICY}; per-label ceiling: " + ", ".join(f"{l}={v['f1']:.3f}" for l, v in _r["per_label"].items()))

In [ ]:
# ---- one experiment ----
RESULTS, TEACHER_LOGITS = [], {}

def count_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

def run_experiment(run):
    cfg = {**COMMON, **run}
    if SMOKE: cfg.update(epochs=1, max_train_windows=16)
    unset = [k for k, v in cfg.items() if v is ...]
    assert not unset, f"run {cfg['name']}: replace the `...` placeholder with a value for {unset}"
    name = cfg["name"]
    out_dir = os.path.join(WORK, name); os.makedirs(out_dir, exist_ok=True)
    section(f"RUN {name}: model {cfg['model']}, method {cfg['method']}")
    log("config:", json.dumps({k: v for k, v in cfg.items() if k not in COMMON or COMMON[k] != v}))
    set_seed(cfg["seed"])
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()

    backbone, info = load_backbone(cfg["model"], cfg["student_layers"])
    log(f"checkpoint {info['folder']} (kind {info['kind']}, layers {info['n_layers']}/{info['full_layers']}, "
        f"prefix_lm={info['prefix_lm']}, unused checkpoint tensors {info['unused_keys']})")
    if cfg["model"] in ("m4", "m5"):                     # same file format: make sure it is the right checkpoint
        assert bool(info["prefix_lm"]) == (cfg["model"] == "m5"), (
            f"{cfg['model']} loaded a checkpoint with prefix_lm={info['prefix_lm']} - wrong MODEL_DIRS['{cfg['model']}']?")
    bpe, enc = get_encodings(info["tokenizer"])
    if info["kind"] != "gru":
        assert cfg["window"] <= info["config"]["max_seq_len"], "window longer than the model's context"
    assert cfg["head"] in ("linear", "crf") and cfg["decode"] in ("argmax", "viterbi")
    model = Tagger(backbone, N_TAGS, cfg["head_dropout"], causal=(cfg["attn"] == "causal" and info["kind"] != "gru"),
                   head=cfg["head"])
    train_ids = TRAIN_IDS + (VAL_IDS if cfg["train_on_val"] else [])
    if cfg["train_on_val"]:
        log(f"train_on_val: training on {len(train_ids)} docs, keeping the last epoch (val scores below are on TRAINING docs)")
    if cfg["head"] == "crf" and cfg["class_weighting"] != "none":
        log("note: class_weighting is not used by the CRF loss")
    groups = setup_method(model, cfg)
    model.to(DEVICE)
    n_total, n_train = count_params(model), count_params(model, True)
    log(f"parameters: total {n_total / 1e6:.2f}M, trainable {n_train / 1e6:.3f}M ({100 * n_train / n_total:.2f}%)")

    teacher = None
    if cfg["method"] == "kd":
        t = cfg["teacher"]
        if t not in TEACHER_LOGITS:
            hits = glob.glob(f"/kaggle/input/**/{t}/teacher_logits.pt", recursive=True)
            assert hits, f"teacher run {t} has not run in this session and no {t}/teacher_logits.pt input was found"
            TEACHER_LOGITS[t] = torch.load(hits[0])
        teacher = TEACHER_LOGITS[t]
        assert all(d in teacher for d in train_ids), "teacher logits do not cover the training docs (train_on_val student?)"
        assert all(len(teacher[d]) == len(enc[d]["ids"]) for d in train_ids), "teacher and student must share the tokenizer"
        log(f"kd: teacher {t}, temperature {cfg['kd_temperature']}, alpha {cfg['kd_alpha']}")

    W, S, P = cfg["window"], cfg["stride"], cfg["preamble_tokens"]
    assert 0 <= P < W and S <= W - P, "need preamble_tokens < window and stride <= window - preamble_tokens"
    train_items = [(d, s) for d in train_ids for s in window_starts(len(enc[d]["ids"]), W - P, S)]
    class_w = None
    if cfg["class_weighting"] == "sqrt_inv":
        cnt = Counter(int(t) for d in train_ids for t in enc[d]["tags"] if t >= 0)
        class_w = torch.tensor([min(10.0, math.sqrt(cnt[0] / max(1, cnt[k]))) for k in range(N_TAGS)], device=DEVICE)
        log("class weights:", [round(x, 2) for x in class_w.tolist()])
    per_epoch = min(len(train_items), cfg["max_train_windows"] or len(train_items))
    steps = math.ceil(per_epoch / cfg["batch_size"] / cfg["grad_accum"]) * cfg["epochs"]
    pg = [dict(params=[p for p in ps if p.ndim >= 2], lr=lr, weight_decay=cfg["weight_decay"]) for _, ps, lr in groups] +          [dict(params=[p for p in ps if p.ndim < 2], lr=lr, weight_decay=0.0) for _, ps, lr in groups]
    opt = torch.optim.AdamW([g for g in pg if g["params"]])
    warm = max(1, int(cfg["warmup_frac"] * steps))
    lr_at = lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / max(1, steps - warm))   # warmup, linear decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16))
    log(f"train windows/epoch {per_epoch} (window {W} = {P} preamble context + {W - P} content, stride {S}), "
        f"optimizer steps {steps}, warmup {warm}")

    best, best_state, history, bad = -1.0, None, [], 0
    t_train = time.time()
    for ep in range(cfg["epochs"]):
        model.train(); t0 = time.time()
        random.shuffle(train_items)
        items = train_items[:per_epoch]
        tot, nb = 0.0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(items), cfg["batch_size"]):
            ids, keep, tags, tl = make_batch(items[bi:bi + cfg["batch_size"]], W, enc, teacher, P)
            ids, tags = ids.to(DEVICE), tags.to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=DEVICE.type == "cuda"):
                lg = model(ids, None if keep is None else keep.to(DEVICE))
            if model.crf is not None: loss = model.crf.nll(lg, tags)
            else: loss = F.cross_entropy(lg.view(-1, N_TAGS), tags.view(-1), weight=class_w, ignore_index=-100)
            if teacher is not None:
                m = tags.view(-1) != -100
                T_ = cfg["kd_temperature"]
                kd = F.kl_div(F.log_softmax(lg.view(-1, N_TAGS)[m] / T_, -1), F.softmax(tl.to(DEVICE).view(-1, N_TAGS)[m] / T_, -1),
                              reduction="batchmean") * T_ * T_
                loss = (1 - cfg["kd_alpha"]) * loss + cfg["kd_alpha"] * kd
            scaler.scale(loss / cfg["grad_accum"]).backward()
            nb += 1; tot += loss.item()
            if nb % cfg["grad_accum"] == 0 or bi + cfg["batch_size"] >= len(items):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], cfg["grad_clip"])
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        vl = doc_logits(model, enc, VAL_IDS, W, S, P=P)
        vm = score(GOLD, predict_spans(vl, enc, DOCS, TRIM_POLICY, KEEP_DOT, model.decoder(cfg["decode"])), VAL_IDS)
        history.append(dict(epoch=ep + 1, train_loss=tot / max(1, nb), val_macro_f1=vm["macro_f1"], val_p=vm["macro_p"],
                            val_r=vm["macro_r"], seconds=time.time() - t0))
        log(f"  epoch {ep + 1:2d}/{cfg['epochs']}: train loss {tot / max(1, nb):.4f} | val strict macro-F1 {vm['macro_f1']:.4f} "
            f"(P {vm['macro_p']:.4f} R {vm['macro_r']:.4f}) | {time.time() - t0:.0f}s")
        if cfg["train_on_val"] or vm["macro_f1"] > best:     # train_on_val: always the latest epoch
            best, bad = vm["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
        else:
            bad += 1
            if cfg["patience"] and bad >= cfg["patience"]:
                log("  early stop"); break
    train_secs = time.time() - t_train
    model.load_state_dict(best_state, strict=False)
    best_ep = history[-1]["epoch"] if cfg["train_on_val"] else max(history, key=lambda h: h["val_macro_f1"])["epoch"]

    # validation report at the best epoch
    dec = model.decoder(cfg["decode"])
    vl = doc_logits(model, enc, VAL_IDS, W, S, P=P)
    vpred = predict_spans(vl, enc, DOCS, TRIM_POLICY, KEEP_DOT, dec)
    vm = score(GOLD, vpred, VAL_IDS)
    eb = error_breakdown(GOLD, vpred, VAL_IDS)
    log(f"best epoch {best_ep}: val strict macro-F1 {vm['macro_f1']:.4f}, P {vm['macro_p']:.4f}, R {vm['macro_r']:.4f} "
        f"(head {cfg['head']}, decode {'viterbi' if dec else 'argmax'})")
    if model.crf is None:                                # same weights, the other decoder (for comparison only)
        other = "argmax" if cfg["decode"] == "viterbi" else "viterbi"
        vo = score(GOLD, predict_spans(vl, enc, DOCS, TRIM_POLICY, KEEP_DOT, model.decoder(other)), VAL_IDS)
        log(f"  same weights with decode={other}: val strict macro-F1 {vo['macro_f1']:.4f}")
    log(f"  {'label':10s} {'P':>6s} {'R':>6s} {'F1':>6s} {'tp':>5s} {'fp':>5s} {'fn':>5s}  error breakdown of gold misses / spurious")
    for l in LABELS:
        v = vm["per_label"][l]
        log(f"  {l:10s} {v['p']:6.3f} {v['r']:6.3f} {v['f1']:6.3f} {v['tp']:5d} {v['fp']:5d} {v['fn']:5d}  {eb[l]}")

    # test predictions
    t0 = time.time()
    tl_ = doc_logits(model, enc, list(TEST), W, S, P=P)
    infer_secs = time.time() - t0
    tpred = predict_spans(tl_, enc, TEST, TRIM_POLICY, KEEP_DOT, dec)
    pred_json = {}
    for d in TEST:                                # every official id, valid labels, 0 <= s < e, no duplicates
        sp = sorted({(int(s), int(e), l) for s, e, l in tpred.get(d, []) if 0 <= s < e <= len(TEST[d]) and l in LABELS})
        pred_json[d] = [[s, e, l, TEST[d][s:e]] for s, e, l in sp]
    assert set(pred_json) == set(TEST)
    with open(os.path.join(out_dir, "prediction.json"), "w", encoding="utf-8") as f:
        json.dump(pred_json, f, ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(out_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out_dir, "prediction.json"), "prediction.json")
    tc = Counter(l for v in pred_json.values() for _, _, l, _ in v)
    log(f"test: {sum(tc.values())} entities predicted ({', '.join(f'{l}={tc[l]}' for l in LABELS)}), inference {infer_secs:.0f}s")

    # teacher logits for later KD runs (merged over windows, train docs)
    if any(r.get("teacher") == name for r in RUNS):
        TEACHER_LOGITS[name] = {d: v.half() for d, v in doc_logits(model, enc, TRAIN_IDS + VAL_IDS, W, S, P=P).items()}
        torch.save(TEACHER_LOGITS[name], os.path.join(out_dir, "teacher_logits.pt"))
        log("saved teacher logits for KD")

    peak = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
    res = dict(name=name, model=cfg["model"], method=cfg["method"], attn=cfg["attn"] if info["kind"] != "gru" else "bigru",
               layers=f"{info['n_layers']}/{info['full_layers']}", total_params=n_total, trainable_params=n_train,
               best_epoch=best_ep, val_macro_f1=vm["macro_f1"], val_macro_p=vm["macro_p"], val_macro_r=vm["macro_r"],
               val_per_label={l: v["f1"] for l, v in vm["per_label"].items()}, train_minutes=train_secs / 60,
               test_inference_seconds=infer_secs, peak_gpu_gb=peak, history=history, config=cfg,
               checkpoint=info["folder"], trim_policy=TRIM_POLICY)
    json.dump(res, open(os.path.join(out_dir, "val_metrics.json"), "w"), indent=1)
    json.dump({d: vpred[d] for d in VAL_IDS}, open(os.path.join(out_dir, "val_predictions.json"), "w"))
    json.dump({"errors": eb}, open(os.path.join(out_dir, "val_error_breakdown.json"), "w"), indent=1)
    if cfg["save_weights"]:
        torch.save(dict(trainable=best_state, config=cfg, backbone_config=info["config"]), os.path.join(out_dir, "trainable_weights.pt"))
    log(f"train {train_secs / 60:.1f} min, peak GPU memory {peak:.2f} GB, outputs in {out_dir}")
    RESULTS.append(res)
    del model, opt, backbone; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return res

In [ ]:
# ---- run everything ----
for run in RUNS:
    if RUN_ONLY and run["name"] not in RUN_ONLY: continue
    try:
        run_experiment(run)
    except Exception as e:                               # one failing run must not lose the others
        import traceback
        log(f"RUN {run['name']} FAILED: {e!r}\n{traceback.format_exc()}")
        gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

In [ ]:
# ---- summary + one zip to download ----
section("SUMMARY")
log(f"{'run':16s} {'model':5s} {'method':7s} {'layers':6s} {'total':>8s} {'trainable':>10s} {'best ep':>7s} "
    f"{'val F1':>7s} {'val P':>6s} {'val R':>6s} {'train min':>9s} {'infer s':>7s} {'peak GB':>7s}")
for r in RESULTS:
    log(f"{r['name']:16s} {r['model']:5s} {r['method']:7s} {r['layers']:6s} {r['total_params'] / 1e6:7.1f}M "
        f"{r['trainable_params'] / 1e6:9.3f}M {r['best_epoch']:7d} {r['val_macro_f1']:7.4f} {r['val_macro_p']:6.4f} "
        f"{r['val_macro_r']:6.4f} {r['train_minutes']:9.1f} {r['test_inference_seconds']:7.0f} {r['peak_gpu_gb']:7.2f}")
log("per-label val F1:")
for r in RESULTS:
    log(f"  {r['name']:16s} " + " ".join(f"{l}={v:.2f}" for l, v in r["val_per_label"].items()))

In [ ]:
# ---- one zip to download ----
json.dump(RESULTS, open(os.path.join(WORK, "phase2_results_summary.json"), "w"), indent=1)
_log.flush()
bundle = os.path.join(WORK, "phase2_results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(WORK, "**", "*"), recursive=True):
        if (f.endswith((".txt", ".json", ".zip", "teacher_logits.pt"))      # teacher logits: reusable as a KD input later
                and not f.endswith("phase2_results_bundle.zip")):
            z.write(f, os.path.relpath(f, WORK))
log(f"bundle: {bundle} ({os.path.getsize(bundle) / 1e6:.1f} MB)")